# P145 — Superar el olvido catastrófico en redes neuronales

## 1. Título y paper

**Paper:** *Overcoming catastrophic forgetting in neural networks*  
**Autoría:** James Kirkpatrick, Razvan Pascanu, Neil Rabinowitz, Joel Veness, Guillaume Desjardins, Andrei A. Rusu, y otros  
**Año y venue:** 2017 · PNAS, 114(13), 3521–3526  
**Nivel:** L3 · **Motor:** `ewc`  
**Ficha completa:** [`P145_ewc`](../../papers/foundational/P145_ewc/README.md)

**Hito:** Frena selectivamente los pesos que importaban para las tareas anteriores y deja libres los demás, con una penalización derivada de la información de Fisher.

- [doi:10.1073/pnas.1611835114](https://doi.org/10.1073/pnas.1611835114)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El olvido catastrófico llevaba treinta años documentado y sin remedio práctico. Reentrenar con todos los datos anteriores resuelve el problema y exige conservarlos, que es justo lo que no siempre se puede.
2. Ejecutar una implementación mínima de la propuesta: Estimar cuánto importa cada peso para lo ya aprendido —aproximando la información de Fisher— y añadir a la pérdida una penalización elástica que tira de esos pesos hacia su valor anterior, con fuerza proporcional a su importancia.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P142
- P02


## 4. Intuición

Si el problema es que aprender lo nuevo mueve los pesos que sostenían lo viejo, la solución es **frenar solo esos** y dejar los demás libres.


## 5. Concepto mínimo

```text
L(θ) = L_B(θ)  +  (λ/2) · Σᵢ Fᵢ · (θᵢ − θ*A,ᵢ)²
                            ↑ importancia del peso i para la tarea A

F ≈ información de Fisher, aproximada por el gradiente al cuadrado
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('ewc', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto cae A sin protección?
2. ¿Cuánto recupera con la penalización?
3. ¿Qué pasa si λ es muy grande?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('ewc', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('ewc', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sin protección, A cae de 0,975 a **0,655** mientras B llega a 0,975. Con λ = 10, A se recupera a **0,97** y B se queda en **0,725**: la media sube de 0,815 a **0,847**. Con λ = 60, ambas colapsan al azar (**0,545** y **0,535**).


## 10. Comentario pedagógico

Fíjate en que **B paga 0,25 puntos** por lo que A recupera. No es una solución sin coste: es un intercambio explícito entre estabilidad y plasticidad, con λ como perilla. Y en esta maqueta el intercambio es duro porque un clasificador lineal tiene un único hiperplano y casi todos sus pesos importan; en una red profunda hay muchísimo más margen libre y ahí es donde el método luce.


## 11. Error o anti-patrón deliberado

Anti-patrón: subir λ hasta que la tarea antigua deje de degradarse.


In [ ]:
print('Con lambda = 60 la tarea antigua deja de degradarse... y queda en 0,545.')
print('La penalizacion domina al gradiente y no se aprende nada, ni lo viejo ni lo nuevo.')
print('Lambda es un compromiso, no un dial de seguridad.')

## 12. Corrección

El barrido de λ, con las dos tareas:


In [ ]:
r = run_paper_lab('ewc', seed=3)['result']
print('A antes de B:', r['exactitud_en_A_antes_de_aprender_B'])
for f in r['por_lambda']:
    print(f)
print('pesos por encima de 0,5 de importancia:', r['pesos_muy_importantes'])

## 13. Desafío guiado

Explica por qué la penalización tiene que ser proporcional a la importancia de cada peso y no uniforme, y qué pasaría si lo fuera.


In [ ]:
r = run_paper_lab('ewc', seed=3)['result']
show(r)

## 14. Desafío autónomo

Si mantienes un modelo que se reentrena periódicamente, mide su rendimiento en una tarea antigua antes y después del último ajuste.


## 15. Evidencia de aprendizaje

Guarda las dos cifras y si el intercambio te compensa.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P145_ewc/README.md) · evaluación formal: [`assessments/papers/P145_ewc.md`](../../assessments/papers/P145_ewc.md)


## 16. Cierre

Aprender sin olvidar y aprender sin centralizar son dos restricciones distintas. La segunda es P146.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
